### There is no correlation between dam_height and the rest of the features, so every model just randomly tries to predict it. Therefore, these NaN values have been left untouched. Here I introduce some attempts

In [ ]:
# Importing modules
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from collections import defaultdict
from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS

In [ ]:
cleaned_detailed_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'detailed_reservoirs_cleaned.csv'
df = pd.read_csv(cleaned_detailed_reservoirs_path)
df.head()

In [ ]:
df['dam_height'].describe()

#### We will see that the lowest MAE achieved is 22, so taking into account that the standard deviation of the 'dam_height' distribution is 29.7, we conclude that it's better to leave the missing values as they are rather than imputing them

### Simplest model: only using numerical features: 

In [ ]:
df_dropped = df[['longitude', 'latitude', 'crest_elevation', 'dam_height']]
training_data = df_dropped[df_dropped['dam_height'].notna()]
testing_data = df_dropped[df_dropped['dam_height'].isna()]
X = training_data.drop(columns=['dam_height'])
y = training_data['dam_height']

kf = KFold(n_splits=10, shuffle=True, random_state=27)
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    model = RandomForestRegressor(random_state=27)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

print(f"Simple Random Forest, Average RMSE: {np.mean(rmse_scores):.3f}")
print(f"Simple Random Forest, Average MAE: {np.mean(mae_scores):.3f}")
print(f"Simple Random Forest, Average R2: {np.mean(r2_scores):.3f}")

### Regular Target Encoding for categorical columns (using only the training set to calculate means, avoiding data leakage)

In [ ]:
def regular_target_encoding(train_df, target_col, categorical_cols):
    df_encoded = train_df.copy()
    for col in categorical_cols:
        means = train_df.groupby(col)[target_col].mean()
        df_encoded[col + '_target_encoded'] = train_df[col].map(means)
    return df_encoded

In [ ]:
df_dropped = df[['longitude', 'latitude', 'crest_elevation', 'basin', 'riverbed', 'province', 'autonomous_community','dam_height']]
training_data = df_dropped[df_dropped['dam_height'].notna()]
testing_data = df_dropped[df_dropped['dam_height'].isna()]
X = training_data.drop(columns=['dam_height'])
y = training_data['dam_height']

categorical_cols = [col for col in X.columns if X[col].dtype == 'object']

kf = KFold(n_splits=10, shuffle=True, random_state=27)
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    if categorical_cols:
        # Fit encoder only on training fold, apply to both train and val
        train_df = X_train.copy()
        train_df['dam_height'] = y_train.values
        X_train_encoded = regular_target_encoding(train_df, 'dam_height', categorical_cols)
        X_train_model = X_train_encoded.drop(columns=categorical_cols + ['dam_height'])

        # For validation, use mapping from train
        X_val_enc = X_val.copy()
        for col in categorical_cols:
            means = y_train.groupby(X_train[col]).mean()
            X_val_enc[col + '_target_encoded'] = X_val[col].map(means)
        X_val_model = X_val_enc.drop(columns=categorical_cols)
    else:
        X_train_model = X_train.copy()
        X_val_model = X_val.copy()

    model = RandomForestRegressor(random_state=27)
    model.fit(X_train_model, y_train)
    y_pred = model.predict(X_val_model)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

print(f"Regular Target Encoding, Average RMSE: {np.mean(rmse_scores):.3f}")
print(f"Regular Target Encoding, Average MAE: {np.mean(mae_scores):.3f}")
print(f"Regular Target Encoding, Average R2: {np.mean(r2_scores):.3f}")

### Regular Target Encoding with less Features

In [ ]:
df_dropped = df[['longitude', 'latitude', 'crest_elevation', 'autonomous_community','dam_height']]
training_data = df_dropped[df_dropped['dam_height'].notna()]
testing_data = df_dropped[df_dropped['dam_height'].isna()]
X = training_data.drop(columns=['dam_height'])
y = training_data['dam_height']

categorical_cols = [col for col in X.columns if X[col].dtype == 'object']

kf = KFold(n_splits=10, shuffle=True, random_state=27)
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    if categorical_cols:
        # Fit encoder only on training fold, apply to both train and val
        train_df = X_train.copy()
        train_df['dam_height'] = y_train.values
        X_train_encoded = regular_target_encoding(train_df, 'dam_height', categorical_cols)
        X_train_model = X_train_encoded.drop(columns=categorical_cols + ['dam_height'])

        # For validation, use mapping from train
        X_val_enc = X_val.copy()
        for col in categorical_cols:
            means = y_train.groupby(X_train[col]).mean()
            X_val_enc[col + '_target_encoded'] = X_val[col].map(means)
        X_val_model = X_val_enc.drop(columns=categorical_cols)
    else:
        X_train_model = X_train.copy()
        X_val_model = X_val.copy()

    model = RandomForestRegressor(random_state=27)
    model.fit(X_train_model, y_train)
    y_pred = model.predict(X_val_model)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

print(f"Regular Target Encoding, Average RMSE: {np.mean(rmse_scores):.3f}")
print(f"Regular Target Encoding, Average MAE: {np.mean(mae_scores):.3f}")
print(f"Regular Target Encoding, Average R2: {np.mean(r2_scores):.3f}")

### Smoothed Target Encoding with less Features

In [ ]:
def smoothed_target_encoding(train_df, target_col, categorical_cols, smoothing=10):
    df_encoded = train_df.copy()
    global_mean = train_df[target_col].mean()
    for col in categorical_cols:
        agg = train_df.groupby(col)[target_col].agg(['mean', 'count'])
        smoothing_factor = 1 / (1 + np.exp(-(agg['count'] - smoothing)))
        smooth = global_mean * (1 - smoothing_factor) + agg['mean'] * smoothing_factor
        df_encoded[col + '_target_encoded'] = train_df[col].map(smooth)
    return df_encoded

In [ ]:
df_dropped = df[['longitude', 'latitude', 'crest_elevation', 'autonomous_community','dam_height']]
training_data = df_dropped[df_dropped['dam_height'].notna()]
testing_data = df_dropped[df_dropped['dam_height'].isna()]
X = training_data.drop(columns=['dam_height'])
y = training_data['dam_height']

categorical_cols = [col for col in X.columns if X[col].dtype == 'object']

kf = KFold(n_splits=5, shuffle=True, random_state=27)
rmse_scores = []
mae_scores = []
r2_scores = []
all_y_true = []
all_y_pred = []

for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    if categorical_cols:
        # Fit encoder only on training fold, apply to both train and val
        train_df = X_train.copy()
        train_df['dam_height'] = y_train.values
        X_train_enc = smoothed_target_encoding(train_df, 'dam_height', categorical_cols)
        X_train_model = X_train_enc.drop(columns=categorical_cols + ['dam_height'])

        # For validation, use mapping from train
        global_mean = y_train.mean()
        X_val_enc = X_val.copy()
        for col in categorical_cols:
            agg = y_train.groupby(X_train[col]).agg(['mean', 'count'])
            smoothing_factor = 1 / (1 + np.exp(-(agg['count'] - 10)))
            smooth = global_mean * (1 - smoothing_factor) + agg['mean'] * smoothing_factor
            X_val_enc[col + '_target_encoded'] = X_val[col].map(smooth)
        X_val_model = X_val_enc.drop(columns=categorical_cols)
    else:
        X_train_model = X_train.copy()
        X_val_model = X_val.copy()

    model = RandomForestRegressor(random_state=27)
    model.fit(X_train_model, y_train)
    y_pred = model.predict(X_val_model)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    all_y_true.extend(y_val)
    all_y_pred.extend(y_pred)

print(f"Smoothed Target Encoding, Average RMSE: {np.mean(rmse_scores):.3f}")
print(f"Smoothed Target Encoding, Average MAE: {np.mean(mae_scores):.3f}")
print(f"Smoothed Target Encoding, Average R2: {np.mean(r2_scores):.3f}")

### Smoothed target encoding with SequentialFeatureSelector for feature selection

In [ ]:
categorical_cols = [col for col in X.columns if X[col].dtype == 'object']
X_base = X.copy()

kf = KFold(n_splits=5, shuffle=True, random_state=27)
rmse_scores = []
mae_scores = []
r2_scores = []

for train_index, val_index in kf.split(X_base):
    X_train, X_val = X_base.iloc[train_index], X_base.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    if categorical_cols:
        # Fit encoder only on training fold, apply to both train and val
        train_df = X_train.copy()
        train_df['dam_height'] = y_train.values
        X_train_enc = smoothed_target_encoding(train_df, 'dam_height', categorical_cols)
        X_train_model = X_train_enc.drop(columns=categorical_cols + ['dam_height'])

        # For validation, use mapping from train
        global_mean = y_train.mean()
        X_val_enc = X_val.copy()
        for col in categorical_cols:
            agg = y_train.groupby(X_train[col]).agg(['mean', 'count'])
            smoothing_factor = 1 / (1 + np.exp(-(agg['count'] - 10)))
            smooth = global_mean * (1 - smoothing_factor) + agg['mean'] * smoothing_factor
            smooth_dict = defaultdict(lambda: global_mean, smooth.to_dict())
            X_val_enc[col + '_target_encoded'] = X_val[col].map(smooth_dict)
        X_val_model = X_val_enc.drop(columns=categorical_cols)
    else:
        X_train_model = X_train.copy()
        X_val_model = X_val.copy()

    # Fit SFS on training fold only
    model = RandomForestRegressor(random_state=27)
    sfs_fold = SequentialFeatureSelector(model, n_features_to_select='auto', direction='forward', scoring='neg_root_mean_squared_error', cv=3, n_jobs=-1)
    sfs_fold.fit(X_train_model, y_train)
    selected_features = X_train_model.columns[sfs_fold.get_support()]

    model_fold = RandomForestRegressor(random_state=27)
    model_fold.fit(X_train_model[selected_features], y_train)
    y_pred = model_fold.predict(X_val_model[selected_features])
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)

print(f"Smoothed Target Encoding with Sequential Feature Selection, Average RMSE: {np.mean(rmse_scores):.3f}")
print(f"Smoothed Target Encoding with Sequential Feature Selection, Average MAE: {np.mean(mae_scores):.3f}")
print(f"Smoothed Target Encoding with Sequential Feature Selection, Average R2: {np.mean(r2_scores):.3f}")